# Results analysis

This notebook collects the results of the 36 configurations x 3 seeds and produces the tables and figures used in the README:

1. metrics for every single run (linear SVM accuracy, Silhouette score, reconstruction loss), saved in `../results/metrics_per_run.csv`
2. summary per configuration (mean and standard deviation over the seeds)
3. 6x6 heatmaps (activation of the first layer x activation of the second layer)
4. average of the effect of each activation, in the first and in the second layer
5. average training and validation loss curves
6. comparison between pairs of configurations with inverted layers (e.g. `selu_elu` vs `elu_selu`)

Metrics are the same of `embedding_evaluation.ipynb`: the SVM is trained on the training embeddings and evaluated on the test embeddings, the Silhouette score is computed on the test embeddings.

Computing the metrics takes some minutes. They are computed only if `metrics_per_run.csv` does not exist (or if `RECOMPUTE = True`).

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from joblib import Parallel, delayed
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, silhouette_score
from IPython.display import display

In [ ]:
EXPERIMENTS_DIR = Path("../experiments")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

METRICS_CSV = RESULTS_DIR / "metrics_per_run.csv"
RECOMPUTE = False

SEEDS = [42, 999, 2026]
ACT_ORDER = ["relu", "lrelu", "elu", "gelu", "selu", "swish"]
ACT_LABELS = {"relu": "ReLU", "lrelu": "Leaky ReLU", "elu": "ELU", "gelu": "GELU", "selu": "SELU", "swish": "Swish"}

## 1) Metrics for every run

In [ ]:
def load_embeddings_and_labels(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    return data["embeddings"].cpu().numpy(), data["labels"].cpu().numpy().ravel()


def evaluate_run(first, second, seed):
    run_dir = EXPERIMENTS_DIR / "4096_{}_1024_{}".format(first, second) / "seed_{}".format(seed)
    X_train, y_train = load_embeddings_and_labels(run_dir / "training_embeddings" / "embeddings.pkl")
    X_test, y_test = load_embeddings_and_labels(run_dir / "test_embeddings" / "embeddings.pkl")

    # Linear probing with SVM (trained on train embeddings, evaluated on test embeddings)
    svm = LinearSVC(dual=False, max_iter=2000, random_state=42).fit(X_train, y_train)
    svm_accuracy = accuracy_score(y_test, svm.predict(X_test))

    # Silhouette score on test embeddings
    silhouette = silhouette_score(X_test, y_test, metric="euclidean")

    with open(run_dir / "results.json") as f:
        results = json.load(f)

    return {
        "first": first, "second": second, "seed": seed,
        "svm_accuracy": svm_accuracy,
        "silhouette": silhouette,
        "test_loss": results["test_metrics"]["loss"],
    }

In [ ]:
if METRICS_CSV.exists() and not RECOMPUTE:
    df_runs = pd.read_csv(METRICS_CSV)
else:
    runs = Parallel(n_jobs=6, verbose=5)(
        delayed(evaluate_run)(first, second, seed)
        for first in ACT_ORDER for second in ACT_ORDER for seed in SEEDS
    )
    df_runs = pd.DataFrame(runs)
    df_runs.to_csv(METRICS_CSV, index=False)

print("{} runs, {} configurations".format(len(df_runs), df_runs.groupby(["first", "second"]).ngroups))
df_runs.head()

## 2) Summary per configuration

Mean and standard deviation (`ddof=1`) over the 3 seeds.

In [ ]:
metrics = ["svm_accuracy", "silhouette", "test_loss"]

summary = df_runs.groupby(["first", "second"])[metrics].agg(["mean", "std"])
summary.columns = ["_".join(col) for col in summary.columns]
summary = summary.reset_index()
summary.to_csv(RESULTS_DIR / "metrics_summary.csv", index=False)

table = pd.DataFrame({
    "Configuration": "4096_" + summary["first"] + "_1024_" + summary["second"],
    "SVM accuracy (%)": summary.apply(lambda r: "{:.2f} ± {:.2f}".format(100 * r.svm_accuracy_mean, 100 * r.svm_accuracy_std), axis=1),
    "Silhouette": summary.apply(lambda r: "{:.4f} ± {:.4f}".format(r.silhouette_mean, r.silhouette_std), axis=1),
    "Test MSE": summary.apply(lambda r: "{:.4f} ± {:.4f}".format(r.test_loss_mean, r.test_loss_std), axis=1),
    "_order": summary["svm_accuracy_mean"],
}).sort_values("_order", ascending=False).drop(columns="_order").reset_index(drop=True)
table.index += 1
table.to_csv(RESULTS_DIR / "ranking_table.csv")

### Top 5 and bottom 5 by SVM accuracy

In [ ]:
display(table.head(5))
display(table.tail(5))

### Full table

In [ ]:
display(table)

## 3) Heatmaps

Rows are the activation of the first layer (4096 units), columns the activation of the second layer (1024 units). The darker the cell the higher the value. Each cell reports mean and standard deviation over the 3 seeds.

In [ ]:
SURFACE, INK, INK_SOFT = "#fcfcfb", "#0b0b0b", "#52514e"
# one-hue (blue) sequential ramp, light -> dark
CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])


def plot_heatmap(metric, title, filename, scale=1.0, decimals=2, cbar_label=""):
    mean = summary.pivot(index="first", columns="second", values=metric + "_mean").loc[ACT_ORDER, ACT_ORDER] * scale
    std = summary.pivot(index="first", columns="second", values=metric + "_std").loc[ACT_ORDER, ACT_ORDER] * scale

    fig, ax = plt.subplots(figsize=(8.4, 6.8), facecolor=SURFACE)
    im = ax.imshow(mean.values, cmap=CMAP)
    norm = im.norm

    for i in range(len(ACT_ORDER)):
        for j in range(len(ACT_ORDER)):
            r, g, b, _ = CMAP(norm(mean.values[i, j]))
            text_color = "white" if 0.299 * r + 0.587 * g + 0.114 * b < 0.5 else INK
            ax.text(j, i - 0.1, "{:.{d}f}".format(mean.values[i, j], d=decimals),
                    ha="center", va="center", color=text_color, fontsize=11, fontweight="bold")
            ax.text(j, i + 0.22, "± {:.{d}f}".format(std.values[i, j], d=decimals),
                    ha="center", va="center", color=text_color, fontsize=8)

    labels = [ACT_LABELS[a] for a in ACT_ORDER]
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, color=INK_SOFT, fontsize=10)
    ax.set_yticklabels(labels, color=INK_SOFT, fontsize=10)
    ax.set_xlabel("Activation of the 2nd layer (1024 units)", color=INK_SOFT, fontsize=12)
    ax.set_ylabel("Activation of the 1st layer (4096 units)", color=INK_SOFT, fontsize=12)
    ax.set_title(title, color=INK, fontsize=14, fontweight="bold", loc="left", pad=14)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.tick_params(length=0)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cbar.set_label(cbar_label, color=INK_SOFT, fontsize=11)
    cbar.ax.tick_params(colors=INK_SOFT, labelsize=9)
    cbar.outline.set_visible(False)

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / filename, dpi=150, facecolor=SURFACE)
    plt.show()

### Linear SVM accuracy

In [ ]:
plot_heatmap("svm_accuracy", "Linear SVM accuracy on test embeddings", "heatmap_svm_accuracy.png",
             scale=100, decimals=2, cbar_label="Accuracy (%)")

### Silhouette score

In [ ]:
plot_heatmap("silhouette", "Silhouette score on test embeddings", "heatmap_silhouette.png",
             decimals=4, cbar_label="Silhouette score")

### Reconstruction loss (MSE on test set, lower is better)

In [ ]:
plot_heatmap("test_loss", "Reconstruction loss (MSE) on the test set", "heatmap_test_loss.png",
             decimals=4, cbar_label="MSE")

## 4) Effect of each activation, averaged over the other layer

Each value is the mean over the 6 configurations (x 3 seeds) in which that activation is used in the first (or in the second) layer.

In [ ]:
rows = []
for act in ACT_ORDER:
    first = df_runs[df_runs["first"] == act]
    second = df_runs[df_runs["second"] == act]
    rows.append({
        "Activation": ACT_LABELS[act],
        "SVM acc. as 1st layer (%)": 100 * first["svm_accuracy"].mean(),
        "SVM acc. as 2nd layer (%)": 100 * second["svm_accuracy"].mean(),
        "Silhouette as 1st layer": first["silhouette"].mean(),
        "Silhouette as 2nd layer": second["silhouette"].mean(),
        "Test MSE as 1st layer": first["test_loss"].mean(),
        "Test MSE as 2nd layer": second["test_loss"].mean(),
    })

marginals = pd.DataFrame(rows).set_index("Activation")
marginals.to_csv(RESULTS_DIR / "marginal_means.csv")
display(marginals.round(4))

## 5) Average loss curves

In [ ]:
train_curves, val_curves = [], []
for first in ACT_ORDER:
    for second in ACT_ORDER:
        for seed in SEEDS:
            with open(EXPERIMENTS_DIR / "4096_{}_1024_{}".format(first, second) / "seed_{}".format(seed) / "results.json") as f:
                res = json.load(f)
            train_curves.append(res["training_losses"])
            val_curves.append(res["validation_losses"])

train_curves, val_curves = np.array(train_curves), np.array(val_curves)
epochs = np.arange(1, train_curves.shape[1] + 1)

fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
for curves, label, color in [(train_curves, "Training", "#2a78d6"), (val_curves, "Validation", "#eb6834")]:
    mean, std = curves.mean(axis=0), curves.std(axis=0, ddof=1)
    ax.plot(epochs, mean, color=color, linewidth=2.2, label=label)
    ax.fill_between(epochs, mean - std, mean + std, color=color, alpha=0.2, linewidth=0)

ax.set_xlabel("Epoch", color=INK_SOFT, fontsize=12)
ax.set_ylabel("MSE", color=INK_SOFT, fontsize=12)
ax.set_title("Average loss over all {} runs (mean ± std)".format(len(train_curves)),
             color=INK, fontsize=14, fontweight="bold", loc="left", pad=14)
ax.grid(True, color="#e3e2de", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color(INK_SOFT)
ax.tick_params(colors=INK_SOFT, labelsize=10)
legend = ax.legend(frameon=False, fontsize=11)
for text in legend.get_texts():
    text.set_color(INK)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "loss_curves.png", dpi=150, facecolor=SURFACE)
plt.show()

## 6) Inverted pairs

For each pair of activations (A, B), accuracy of `A` in the first layer and `B` in the second, against the inverted configuration. The last column is the difference between the two means: compare it with the standard deviations, not with zero.

In [ ]:
rows = []
for i, a in enumerate(ACT_ORDER):
    for b in ACT_ORDER[i + 1:]:
        ab = summary[(summary["first"] == a) & (summary["second"] == b)].iloc[0]
        ba = summary[(summary["first"] == b) & (summary["second"] == a)].iloc[0]
        rows.append({
            "Pair": "{} / {}".format(ACT_LABELS[a], ACT_LABELS[b]),
            "A→B acc. (%)": "{:.2f} ± {:.2f}".format(100 * ab.svm_accuracy_mean, 100 * ab.svm_accuracy_std),
            "B→A acc. (%)": "{:.2f} ± {:.2f}".format(100 * ba.svm_accuracy_mean, 100 * ba.svm_accuracy_std),
            "Difference (pp)": 100 * (ab.svm_accuracy_mean - ba.svm_accuracy_mean),
        })

inverted = pd.DataFrame(rows)
inverted.to_csv(RESULTS_DIR / "inverted_pairs.csv", index=False)
display(inverted.round(2))